In [18]:
import pandas as pd
import numpy as np

In [19]:
# Load the data and create engineered features
df = pd.read_csv("../../data/data.csv")
# Convert date columns to datetime
df[["observation_date", "date_of_introduction"]] = df[
    ["observation_date", "date_of_introduction"]
].apply(pd.to_datetime)
# create number of days since introduction feature
df["days_since_introduction"] = (
    df["observation_date"] - df["date_of_introduction"]
).dt.days
df["initial_growth_potential"] = df["initial_female_count"] * df["leaf_area_cm2"]
df.head()

,treatment_id,date_of_introduction,observation_date,initial_female_count,leaf_area_cm2,population_density,replicate_id,cumulative_mite_count,number_of_eggs,no_of_eggs_hatched,male_count,female_count,days_since_introduction,initial_growth_potential
0,T1R1,2019-07-31,2019-08-01,1,0.5,2.0,1,0,0,0,0,0,1,0.5
1,T1R1,2019-07-31,2019-08-02,1,0.5,2.0,1,0,0,0,0,0,2,0.5
2,T1R1,2019-07-31,2019-08-03,1,0.5,2.0,1,2,3,2,0,2,3,0.5
3,T1R1,2019-07-31,2019-08-04,1,0.5,2.0,1,6,4,4,0,4,4,0.5
4,T1R1,2019-07-31,2019-08-05,1,0.5,2.0,1,9,4,3,0,3,5,0.5


In [20]:
df.drop(
    columns=["treatment_id","date_of_introduction", "observation_date"],
    axis=1,
    inplace=True,
)

In [24]:
from imblearn.over_sampling import SMOTE

# Separate features and target
X = df.drop('cumulative_mite_count', axis=1)
y = df['cumulative_mite_count']

# Apply SMOTE
smote = SMOTE(random_state=42,k_neighbors=min(len(y[y == y.unique()[0]]) - 1, 5))
X_resampled, y_resampled = smote.fit_resample(X, y)

# Combine back into a single DataFrame
augmented_data = pd.DataFrame(X_resampled, columns=X.columns)
augmented_data['cumulative_mite_count'] = y_resampled

# Round non-Density columns to nearest whole number
for column in augmented_data.columns:
    if column not in ['leaf_area_cm2','population_density','initial_growth_potential']:
        augmented_data[column] = augmented_data[column].astype(int) # Ensure values are integers

ValueError: Expected n_neighbors <= n_samples_fit, but n_neighbors = 6, n_samples_fit = 5, n_samples = 5

In [25]:
from imblearn.over_sampling import SMOTE

# Separate features and target
X = df.drop('cumulative_mite_count', axis=1)
y = df['cumulative_mite_count']

# Calculate appropriate k_neighbors
# Find the smallest class size
class_counts = y.value_counts()
min_class_size = class_counts.min()

# Set k_neighbors to be at most min_class_size - 1
k_neighbors = min(min_class_size - 1, 5)

# Ensure k_neighbors is at least 1
k_neighbors = max(k_neighbors, 1)

# Apply SMOTE
smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Combine back into a single DataFrame
augmented_data = pd.DataFrame(X_resampled, columns=X.columns)
augmented_data['cumulative_mite_count'] = y_resampled

# Round non-Density columns to nearest whole number
for column in augmented_data.columns:
    if column not in ['leaf_area_cm2','population_density','initial_growth_potential']:
        augmented_data[column] = augmented_data[column].astype(int) # Ensure values are integers

ValueError: Expected n_neighbors <= n_samples_fit, but n_neighbors = 2, n_samples_fit = 1, n_samples = 1